## 06. Feature Selection

This notebook preserves the original screening and selection alternatives using common training data in its original scale. No distributions or encoders are required from previous notebooks. Decisions here describe a local experimental feature set, not a change to load_stage("common"). Selection based on target associations or estimated statistics must be evaluated within training folds in Modeling.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker

df_before = load_stage("common")
df_hp = df_before.copy()
tracker = DataTracker.load(df_before)
profile = tracker.profile
display(profile)


2026-10-12 02:06:19.178 | INFO     | house_prices_mlops.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
0,Id,int64,identifier,1168,0,0.000000,0.0,0.000000,1.0,1460.0,730.904966,732.5,425.369088,0.000920,-1.215850
1,MSSubClass,string,nominal,15,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MSZoning,object,nominal,5,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LotFrontage,float64,continuous,107,217,18.578767,0.0,0.000000,21.0,313.0,70.343849,70.0,24.897021,2.406787,19.454319
4,LotArea,int64,continuous,890,0,0.000000,0.0,0.000000,1300.0,215245.0,10689.642123,9600.0,10759.366198,11.958088,186.444082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,HasBasement,int64,None,2,0,0.000000,28.0,2.397260,0.0,1.0,0.976027,1.0,0.153029,-6.232060,36.901760
89,HasPool,int64,None,2,0,0.000000,1162.0,99.486301,0.0,1.0,0.005137,0.0,0.071519,13.862369,190.491446
90,HasFireplace,int64,None,2,0,0.000000,547.0,46.832192,0.0,1.0,0.531678,1.0,0.499209,-0.127131,-1.987244
91,IsRemodeled,int64,None,2,0,0.000000,601.0,51.455479,0.0,1.0,0.485445,0.0,0.500002,0.058319,-2.000027


### Low Variance

In [2]:
#LOW VARIANCE
#Variables que casi no cambian

low_variance_analysis = []

for var in df_hp.columns:

    if var in ('Id', 'SalePrice'):
        continue

    counts = df_hp[var].value_counts(
        normalize=True,
        dropna=False
    )

    low_variance_analysis.append({
        'var': var,
        'unique_count': df_hp[var].nunique(dropna=False),
        'dominant_value': counts.index[0],
        'dominant_pct': counts.iloc[0] * 100
    })

low_variance_analysis = pd.DataFrame(
    low_variance_analysis
).sort_values(
    'dominant_pct',
    ascending=False
)

display(low_variance_analysis.head(20))

,var,unique_count,dominant_value,dominant_pct
8,Utilities,2,AllPub,99.914384
4,Street,2,Pave,99.657534
70,PoolArea,7,0,99.486301
87,HasPool,2,0,99.486301
71,PoolQC,4,None,99.486301
13,Condition2,8,Norm,99.058219
21,RoofMatl,7,CompShg,98.373288
68,3SsnPorch,17,0,98.287671
44,LowQualFinSF,20,0,98.202055
38,Heating,6,GasA,97.602740


In [3]:
low_variance_candidates = low_variance_analysis[
    low_variance_analysis['dominant_pct'] >= 95
]

display(low_variance_candidates)

,var,unique_count,dominant_value,dominant_pct
8,Utilities,2,AllPub,99.914384
4,Street,2,Pave,99.657534
70,PoolArea,7,0,99.486301
87,HasPool,2,0,99.486301
71,PoolQC,4,None,99.486301
13,Condition2,8,Norm,99.058219
21,RoofMatl,7,CompShg,98.373288
68,3SsnPorch,17,0,98.287671
44,LowQualFinSF,20,0,98.202055
38,Heating,6,GasA,97.602740


In [4]:
low_var_names = low_variance_candidates['var'].tolist()

numeric_low_var = [
    var for var in low_var_names
    if pd.api.types.is_numeric_dtype(df_hp[var])
]

target_corr = (
     df_hp[numeric_low_var + ['SalePrice']]
    .corr()['SalePrice']
    .drop('SalePrice')
    .sort_values(key=abs, ascending=False)
)

display(target_corr)

HasBasement     0.155615
KitchenAbvGr   -0.142785
HasPool         0.117439
PoolArea        0.115630
3SsnPorch       0.051532
MiscVal        -0.020179
LowQualFinSF   -0.011189
Name: SalePrice, dtype: float64

In [5]:
categorical_low_var = [
    var for var in low_variance_candidates['var']
    if not pd.api.types.is_numeric_dtype(df_hp[var])
]

for var in categorical_low_var:
    display(
        df_hp.groupby(var)['SalePrice']
        .agg(['count', 'mean', 'median'])
        .sort_values('mean', ascending=False)
    )

,count,mean,median
Utilities,,,
AllPub,1167,181479.195373,165000.0
NoSeWa,1,137500.000000,137500.0


,count,mean,median
Street,,,
Pave,1164,181511.658935,165000.0
Grvl,4,161037.500000,152600.0


,count,mean,median
PoolQC,,,
Ex,2,490000.000000,490000.0
Gd,2,217485.000000,217485.0
Fa,2,215500.000000,215500.0
None,1162,180789.802926,164995.0


,count,mean,median
Condition2,,,
PosA,1,325000.000000,325000.0
PosN,2,284875.000000,284875.0
RRAe,1,190000.000000,190000.0
Norm,1157,181489.685393,165000.0
RRAn,1,136905.000000,136905.0
Feedr,3,126833.333333,128000.0
RRNn,1,125000.000000,125000.0
Artery,2,106500.000000,106500.0


,count,mean,median
RoofMatl,,,
WdShngl,4,326625.000000,331500.0
WdShake,3,226000.000000,228000.0
Tar&Grv,9,192385.555556,185000.0
CompShg,1149,180792.646649,164700.0
Metal,1,180000.000000,180000.0
ClyTile,1,160000.000000,160000.0
Roll,1,137000.000000,137000.0


,count,mean,median
Heating,,,
GasA,1140,182659.247368,165750.0
GasW,15,168858.600000,133900.0
OthW,2,125750.000000,125750.0
Wall,4,92100.000000,91450.0
Grav,6,77816.666667,79500.0
Floor,1,72500.000000,72500.0


,count,mean,median
MiscFeature,,,
TenC,1,250000.000000,250000.0
None,1122,182354.163102,165450.0
Gar2,2,170750.000000,170750.0
Shed,41,159581.707317,157900.0
Othr,2,94000.000000,94000.0


In [6]:
tracker.add_finding(
    key="feature_selection_low_variance", var="multiple", stage="DP - Feature Selection",
    finding=f"El análisis identificó {len(low_variance_candidates)} variables con una categoría dominante de al menos 95%; requieren evaluar redundancia y relación con el target antes de seleccionar columnas.",
)
tracker.add_finding(
    key="pool_features_redundancy", var="HasPool, PoolArea, PoolQC", stage="DP - Feature Selection",
    finding="Las variables de piscina con baja variación pueden representar información redundante.",
)


'pool_features_redundancy'

### Redundancy

In [7]:

# Redundancy  
# Variables que representan información muy similar
# o que nosotros mismos duplicamos con Feature Engineering


numeric_cols = df_hp.select_dtypes(
    include='number'
).columns.drop(['SalePrice', 'Id'], errors='ignore')

corr_matrix = df_hp[numeric_cols].corr().abs()

high_corr_pairs = (
    corr_matrix
    .where(
        np.triu(
            np.ones(corr_matrix.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

high_corr_pairs.columns = [
    'var_1',
    'var_2',
    'correlation'
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs['correlation'] >= 0.80
].sort_values(
    'correlation',
    ascending=False
)

display(high_corr_pairs)

,var_1,var_2,correlation
208,YearBuilt,HouseAge,0.999072
818,GarageYrBlt,GarageAge,0.998565
250,YearRemodAdd,RemodAge,0.997964
971,PoolArea,HasPool,0.987966
584,GrLivArea,QualGrLivArea,0.927383
802,Fireplaces,HasFireplace,0.899667
1050,TotalSF,QualGrLivArea,0.887800
827,GarageCars,GarageArea,0.882550
576,GrLivArea,TotalSF,0.878406
415,TotalBsmtSF,1stFlrSF,0.827382


In [8]:
tracker.add_finding(
    key="feature_selection_high_correlation", var="multiple", stage="DP - Feature Selection",
    finding="Se identificaron pares de variables con correlación elevada; la selección requiere revisar las variables temporales, binarias y agregadas junto con los demás criterios.",
)
temporal_pairs = [("YearBuilt", "HouseAge"), ("YearRemodAdd", "RemodAge"), ("GarageYrBlt", "GarageAge")]
temporal_review = pd.DataFrame([
    {"original": left, "derived": right, "correlation": df_hp[[left, right]].corr().iloc[0, 1]}
    for left, right in temporal_pairs
])
display(temporal_review)
tracker.add_finding(
    key="temporal_features_redundancy", var="YearBuilt, YearRemodAdd, GarageYrBlt", stage="DP - Feature Selection",
    finding="Las variables de año original y las edades derivadas requieren una revisión de redundancia; las correlaciones se muestran en temporal_review.",
)


,original,derived,correlation
0,YearBuilt,HouseAge,-0.999072
1,YearRemodAdd,RemodAge,-0.997964
2,GarageYrBlt,GarageAge,-0.998565


'temporal_features_redundancy'

### Relationship with Target

In [9]:
# Relationship with Target
# Numéricas → correlación / asociación
# Categóricas → asociación con SalePrice

In [10]:
# Variables numéricas excepto target
numeric_features = [
    var for var in df_hp.select_dtypes(include='number').columns
    if var not in ('Id', 'SalePrice')
]

target_relationship = (
    df_hp[numeric_features + ['SalePrice']]
    .corr()['SalePrice']
    .drop('SalePrice')
    .sort_values(key=abs, ascending=False)
)

display(target_relationship)

QualGrLivArea     0.816810
OverallQual       0.785555
TotalSF           0.763242
GrLivArea         0.695652
GarageCars        0.640991
GarageArea        0.624139
TotalBathrooms    0.621280
TotalBsmtSF       0.597766
1stFlrSF          0.587883
FullBath          0.552546
TotRmsAbvGrd      0.520388
YearBuilt         0.516501
HouseAge         -0.516428
RemodAge         -0.509671
YearRemodAdd      0.508593
GarageAge        -0.480389
GarageYrBlt       0.480351
HasFireplace      0.470568
MasVnrArea        0.459123
Fireplaces        0.457549
TotalPorchSF      0.388528
BsmtFinSF1        0.359460
LotFrontage       0.330066
WoodDeckSF        0.329843
2ndFlrSF          0.314030
OpenPorchSF       0.299969
HalfBath          0.280481
LotArea           0.266204
HasGarage         0.247735
BsmtFullBath      0.226346
BsmtUnfSF         0.222487
BedroomAbvGr      0.156211
HasBasement       0.155615
EnclosedPorch    -0.149532
KitchenAbvGr     -0.142785
ScreenPorch       0.119172
HasPool           0.117439
P

In [11]:
#Categoricas

import numpy as np
import pandas as pd

def correlation_ratio(categories, values):
    categories = pd.Series(categories)
    values = pd.Series(values)

    codes, _ = pd.factorize(categories)

    overall_mean = values.mean()

    numerator = 0
    denominator = ((values - overall_mean) ** 2).sum()

    for code in np.unique(codes):
        group = values[codes == code]

        numerator += len(group) * (
            group.mean() - overall_mean
        ) ** 2

    if denominator == 0:
        return 0

    return np.sqrt(numerator / denominator)

In [12]:
categorical_features = df_hp.select_dtypes(
    include=['object', 'category', 'string']
).columns

categorical_target_relationship = pd.DataFrame({
    'var': categorical_features,
    'eta': [
        correlation_ratio(
            df_hp[var],
            df_hp['SalePrice']
        )
        for var in categorical_features
    ]
}).sort_values(
    'eta',
    ascending=False
)

display(categorical_target_relationship)

,var,eta
9,Neighborhood,0.729316
31,KitchenQual,0.679274
22,BsmtQual,0.678592
19,ExterQual,0.671465
35,GarageFinish,0.553277
33,FireplaceQu,0.532980
0,MSSubClass,0.500113
21,Foundation,0.499244
34,GarageType,0.498810
28,HeatingQC,0.459482


### Multicollinearity (VIF)

VIF calculations can emit rank-deficiency or poor-conditioning warnings because aggregate and source columns have exact linear dependencies. Infinite or unstable values require interpretation, not automatic removal. Initial original calculations omit an intercept; later review includes one, so those tables are not strictly comparable. Rows with missing numeric values are excluded only from the VIF calculation, not from the dataset.

In [13]:

# Multicollinearity
# Variables predictoras demasiado relacionadas entre sí

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Solo predictores numéricos
X_vif = df_hp.select_dtypes(include='number').drop(
    columns=['SalePrice']
).copy()

# Quitamos Id porque no es predictor real
X_vif = X_vif.drop(columns=['Id'], errors='ignore')

# Seguridad
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)
X_vif = X_vif.dropna()

vif_results = pd.DataFrame({
    'var': X_vif.columns,
    'VIF': [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
}).sort_values(
    'VIF',
    ascending=False
)

display(vif_results)

/tmp/ipykernel_486758/402000132.py:21: UserWarning: The design matrix is poorly conditioned (condition number=2.26e+16). VIF calculations may be numerically unstable.
  variance_inflation_factor(X_vif.values, i)
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix i

,var,VIF
4,YearBuilt,1.000800e+15
12,2ndFlrSF,1.000800e+15
11,1stFlrSF,1.000800e+15
10,TotalBsmtSF,1.000800e+15
9,BsmtUnfSF,1.000800e+15
8,BsmtFinSF2,1.000800e+15
7,BsmtFinSF1,1.000800e+15
13,LowQualFinSF,1.000800e+15
14,GrLivArea,1.000800e+15
28,EnclosedPorch,1.000800e+15


In [14]:
tracker.add_finding(
    key="feature_selection_vif", var="multiple", stage="DP - Feature Selection",
    finding=f"El análisis VIF identificó {int(np.isinf(vif_results['VIF']).sum())} valores infinitos y {int(vif_results['VIF'].gt(10).sum())} predictores con VIF mayor que 10; requieren revisar multicolinealidad.",
)


'feature_selection_vif'

### Final Selection Alternatives

In [15]:
# Final Selection
#    ├── Keep
#    ├── Drop
#    └── Pending / evaluar con modelo

In [16]:
display(
    tracker.findings[
        tracker.findings['stage'] == 'DP - Feature Selection'
    ]
)

,key,var,stage,finding,status,resolution
24,feature_selection_low_variance,multiple,DP - Feature Selection,El análisis identificó 14 variables con una ca...,pending,None
25,pool_features_redundancy,"HasPool, PoolArea, PoolQC",DP - Feature Selection,Las variables de piscina con baja variación pu...,pending,None
26,feature_selection_high_correlation,multiple,DP - Feature Selection,Se identificaron pares de variables con correl...,pending,None
27,temporal_features_redundancy,"YearBuilt, YearRemodAdd, GarageYrBlt",DP - Feature Selection,Las variables de año original y las edades der...,pending,None
28,feature_selection_vif,multiple,DP - Feature Selection,El análisis VIF identificó 0 valores infinitos...,pending,None


In [17]:
display(tracker.findings.loc[tracker.findings["key"].eq("feature_selection_low_variance")])


,key,var,stage,finding,status,resolution
24,feature_selection_low_variance,multiple,DP - Feature Selection,El análisis identificó 14 variables con una ca...,pending,None


In [18]:
drop_low_variance = [
    'Utilities',
    'Street',
    '3SsnPorch',
    'LowQualFinSF',
    'MiscVal'
]

keep_low_variance = [
    'Heating',
    'HasBasement',
    'KitchenAbvGr'
]

review_low_variance = [
    'Condition2',
    'RoofMatl',
    'MiscFeature'
]

deferred_low_variance = [
    'HasPool',
    'PoolArea',
    'PoolQC'
]

In [19]:
for var in drop_low_variance:
    tracker.add_plan(
        var=var,
        stage='DP - Feature Selection',
        action='drop',
        method='low_variance',
        params={
            'reason': 'near_constant_and_low_target_relationship'
        }
    )
tracker.resolve_finding(
    key='feature_selection_low_variance',
    resolution=(
        'Se revisó la baja variación y se definieron alternativas locales de conservar, eliminar o evaluar columnas. Las decisiones deberán comprobarse en los experimentos.'
    )
)

In [20]:
display(
    tracker.findings[
        tracker.findings['stage'] == 'DP - Feature Selection'
    ]
)

,key,var,stage,finding,status,resolution
24,feature_selection_low_variance,multiple,DP - Feature Selection,El análisis identificó 14 variables con una ca...,resolved,Se revisó la baja variación y se definieron al...
25,pool_features_redundancy,"HasPool, PoolArea, PoolQC",DP - Feature Selection,Las variables de piscina con baja variación pu...,pending,None
26,feature_selection_high_correlation,multiple,DP - Feature Selection,Se identificaron pares de variables con correl...,pending,None
27,temporal_features_redundancy,"YearBuilt, YearRemodAdd, GarageYrBlt",DP - Feature Selection,Las variables de año original y las edades der...,pending,None
28,feature_selection_vif,multiple,DP - Feature Selection,El análisis VIF identificó 0 valores infinitos...,pending,None


In [21]:
for var in ['HasPool', 'PoolQC']:
    tracker.add_plan(
        var=var,
        stage='DP - Feature Selection',
        action='drop',
        method='redundancy',
        params={
            'reason': 'redundant_pool_feature',
            'retained_feature': 'PoolArea'
        }
    )

In [22]:
tracker.resolve_finding(
    key='pool_features_redundancy',
    resolution=(
        'En la alternativa local se conserva PoolArea y se propone eliminar HasPool y PoolQC por redundancia.'
    )
)

In [23]:
display(
    tracker.findings[
        tracker.findings['stage'] == 'DP - Feature Selection'
    ]
)

,key,var,stage,finding,status,resolution
24,feature_selection_low_variance,multiple,DP - Feature Selection,El análisis identificó 14 variables con una ca...,resolved,Se revisó la baja variación y se definieron al...
25,pool_features_redundancy,"HasPool, PoolArea, PoolQC",DP - Feature Selection,Las variables de piscina con baja variación pu...,resolved,En la alternativa local se conserva PoolArea y...
26,feature_selection_high_correlation,multiple,DP - Feature Selection,Se identificaron pares de variables con correl...,pending,None
27,temporal_features_redundancy,"YearBuilt, YearRemodAdd, GarageYrBlt",DP - Feature Selection,Las variables de año original y las edades der...,pending,None
28,feature_selection_vif,multiple,DP - Feature Selection,El análisis VIF identificó 0 valores infinitos...,pending,None


In [24]:
tracker.resolve_finding(
    key='feature_selection_high_correlation',
    resolution=(
        'Se revisaron las correlaciones elevadas; la selección no se basó únicamente en correlación y se contrastó con VIF.'
    )
)

In [25]:
display(
    tracker.findings[
        tracker.findings['stage'] == 'DP - Feature Selection'
    ]
)

,key,var,stage,finding,status,resolution
24,feature_selection_low_variance,multiple,DP - Feature Selection,El análisis identificó 14 variables con una ca...,resolved,Se revisó la baja variación y se definieron al...
25,pool_features_redundancy,"HasPool, PoolArea, PoolQC",DP - Feature Selection,Las variables de piscina con baja variación pu...,resolved,En la alternativa local se conserva PoolArea y...
26,feature_selection_high_correlation,multiple,DP - Feature Selection,Se identificaron pares de variables con correl...,resolved,Se revisaron las correlaciones elevadas; la se...
27,temporal_features_redundancy,"YearBuilt, YearRemodAdd, GarageYrBlt",DP - Feature Selection,Las variables de año original y las edades der...,pending,None
28,feature_selection_vif,multiple,DP - Feature Selection,El análisis VIF identificó 0 valores infinitos...,pending,None


In [26]:
for var in [
    'YearBuilt',
    'YearRemodAdd',
    'GarageYrBlt'
]:
    tracker.add_plan(
        var=var,
        stage='DP - Feature Selection',
        action='drop',
        method='redundancy',
        params={
            'reason': 'replaced_by_derived_age_feature'
        }
    )

In [27]:
tracker.resolve_finding(
    key='temporal_features_redundancy',
    resolution=(
        'En la alternativa local se conservan las edades derivadas y se propone eliminar los años originales.'
    )
)

In [28]:
display(
    tracker.findings[
        tracker.findings['stage'] == 'DP - Feature Selection'
    ]
)

,key,var,stage,finding,status,resolution
24,feature_selection_low_variance,multiple,DP - Feature Selection,El análisis identificó 14 variables con una ca...,resolved,Se revisó la baja variación y se definieron al...
25,pool_features_redundancy,"HasPool, PoolArea, PoolQC",DP - Feature Selection,Las variables de piscina con baja variación pu...,resolved,En la alternativa local se conserva PoolArea y...
26,feature_selection_high_correlation,multiple,DP - Feature Selection,Se identificaron pares de variables con correl...,resolved,Se revisaron las correlaciones elevadas; la se...
27,temporal_features_redundancy,"YearBuilt, YearRemodAdd, GarageYrBlt",DP - Feature Selection,Las variables de año original y las edades der...,resolved,En la alternativa local se conservan las edade...
28,feature_selection_vif,multiple,DP - Feature Selection,El análisis VIF identificó 0 valores infinitos...,pending,None


### VIF Review After Proposed Drops

In [29]:
planned_drops = tracker.plans.loc[
    (tracker.plans['stage'] == 'DP - Feature Selection') &
    (tracker.plans['action'] == 'drop'),
    'var'
].tolist()

planned_drops

['Utilities',
 'Street',
 '3SsnPorch',
 'LowQualFinSF',
 'MiscVal',
 'HasPool',
 'PoolQC',
 'YearBuilt',
 'YearRemodAdd',
 'GarageYrBlt']

In [30]:
df_vif_review = df_hp.drop(
    columns=planned_drops,
    errors='ignore'
)

X_vif_review = df_vif_review.select_dtypes(
    include='number'
).drop(
    columns=['SalePrice', 'Id'],
    errors='ignore'
).copy()

X_vif_review = X_vif_review.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

vif_review = pd.DataFrame({
    'var': X_vif_review.columns,
    'VIF': [
        variance_inflation_factor(
            X_vif_review.values,
            i
        )
        for i in range(X_vif_review.shape[1])
    ]
}).sort_values(
    'VIF',
    ascending=False
)

display(vif_review)

/tmp/ipykernel_486758/1603976906.py:21: UserWarning: The design matrix is poorly conditioned (condition number=1.45e+16). VIF calculations may be numerically unstable.
  variance_inflation_factor(
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficien

,var,VIF
6,BsmtFinSF2,1.000800e+15
7,BsmtUnfSF,1.000800e+15
5,BsmtFinSF1,1.000800e+15
14,FullBath,1.000800e+15
12,BsmtFullBath,1.000800e+15
13,BsmtHalfBath,1.000800e+15
11,GrLivArea,1.000800e+15
8,TotalBsmtSF,1.000800e+15
32,TotalSF,1.000800e+15
15,HalfBath,1.000800e+15


In [31]:
for var in [
    'FullBath',
    'HalfBath',
    'BsmtFullBath',
    'BsmtHalfBath',
    'HasGarage'
]:
    tracker.add_plan(
        var=var,
        stage='DP - Feature Selection',
        action='drop',
        method='multicollinearity',
        params={
            'reason': 'redundant_with_engineered_feature'
        }
    )

In [32]:
planned_drops = tracker.plans.loc[
    (tracker.plans['stage'] == 'DP - Feature Selection') &
    (tracker.plans['action'] == 'drop'),
    'var'
].tolist()

print(len(planned_drops))
print(planned_drops)

15
['Utilities', 'Street', '3SsnPorch', 'LowQualFinSF', 'MiscVal', 'HasPool', 'PoolQC', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'HasGarage']


In [33]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_vif_review = (
    df_hp
    .drop(columns=planned_drops, errors='ignore')
    .select_dtypes(include='number')
    .drop(columns=['SalePrice', 'Id'], errors='ignore')
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# IMPORTANTE: agregar intercepto
X_vif_const = add_constant(X_vif_review)

vif_review = pd.DataFrame({
    'var': X_vif_const.columns,
    'VIF': [
        variance_inflation_factor(X_vif_const.values, i)
        for i in range(X_vif_const.shape[1])
    ]
})

# La constante no es una feature
vif_review = (
    vif_review[
        vif_review['var'] != 'const'
    ]
    .sort_values('VIF', ascending=False)
    .reset_index(drop=True)
)

display(vif_review)

/tmp/ipykernel_486758/197088611.py:19: UserWarning: The design matrix is poorly conditioned (condition number=1.48e+16). VIF calculations may be numerically unstable.
  variance_inflation_factor(X_vif_const.values, i)
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops/.venv/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design ma

,var,VIF
0,TotalBsmtSF,1.000800e+15
1,BsmtFinSF1,1.000800e+15
2,BsmtFinSF2,1.000800e+15
3,GrLivArea,1.000800e+15
4,TotalSF,1.000800e+15
5,BsmtUnfSF,1.000800e+15
6,2ndFlrSF,1.093236e+02
7,1stFlrSF,9.447021e+01
8,QualGrLivArea,4.979206e+01
9,TotalPorchSF,2.431023e+01


In [34]:
for var in [
    'TotalBsmtSF',
    'GrLivArea',
    '1stFlrSF',
    '2ndFlrSF'
]:
    tracker.add_plan(
        var=var,
        stage='DP - Feature Selection',
        action='drop',
        method='multicollinearity',
        params={
            'reason': 'redundant_with_engineered_aggregate_features'
        }
    )

In [35]:
planned_drops = tracker.plans.loc[
    (tracker.plans['stage'] == 'DP - Feature Selection') &
    (tracker.plans['action'] == 'drop'),
    'var'
].tolist()

print(len(planned_drops))
print(planned_drops)

19
['Utilities', 'Street', '3SsnPorch', 'LowQualFinSF', 'MiscVal', 'HasPool', 'PoolQC', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'HasGarage', 'TotalBsmtSF', 'GrLivArea', '1stFlrSF', '2ndFlrSF']


In [36]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_vif_review = (
    df_hp
    .drop(columns=planned_drops, errors='ignore')
    .select_dtypes(include='number')
    .drop(columns=['SalePrice', 'Id'], errors='ignore')
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# IMPORTANTE: agregar intercepto
X_vif_const = add_constant(X_vif_review)

vif_review = pd.DataFrame({
    'var': X_vif_const.columns,
    'VIF': [
        variance_inflation_factor(X_vif_const.values, i)
        for i in range(X_vif_const.shape[1])
    ]
})

# La constante no es una feature
vif_review = (
    vif_review[
        vif_review['var'] != 'const'
    ]
    .sort_values('VIF', ascending=False)
    .reset_index(drop=True)
)

display(vif_review)

,var,VIF
0,TotalSF,77.239350
1,QualGrLivArea,47.293229
2,BsmtUnfSF,26.792537
3,BsmtFinSF1,26.692606
4,TotalPorchSF,24.288364
5,WoodDeckSF,15.841130
6,OverallQual,10.292901
7,Fireplaces,6.728396
8,HasFireplace,6.563856
9,HouseAge,6.478774


In [37]:
tracker.resolve_finding(
    key='feature_selection_vif',
    resolution=(
        'Se evaluó la multicolinealidad tras las eliminaciones propuestas. TotalSF y QualGrLivArea se conservaron como alternativas pendientes de validación con modelos.'
    )
)

### Local Selection

In [38]:
#Ejecutamos el plan y eliminamos los 19 vars

planned_drops = tracker.plans.loc[
    (tracker.plans['stage'] == 'DP - Feature Selection') &
    (tracker.plans['action'] == 'drop'),
    'var'
].tolist()

print(f'Features a eliminar: {len(planned_drops)}')
print(planned_drops)

Features a eliminar: 19
['Utilities', 'Street', '3SsnPorch', 'LowQualFinSF', 'MiscVal', 'HasPool', 'PoolQC', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'HasGarage', 'TotalBsmtSF', 'GrLivArea', '1stFlrSF', '2ndFlrSF']


In [39]:
df_fs = df_hp.drop(
    columns=planned_drops,
    errors='raise'
).copy()

In [40]:
print('Antes:', df_hp.shape)
print('Después:', df_fs.shape)
print('Eliminadas:', df_hp.shape[1] - df_fs.shape[1])

Antes: (1168, 93)
Después: (1168, 74)
Eliminadas: 19


In [41]:
remaining_drops = [
    var for var in planned_drops
    if var in df_fs.columns
]

print('Variables planeadas que siguen presentes:', remaining_drops)

Variables planeadas que siguen presentes: []


In [42]:
mask = (
    (tracker.plans['stage'] == 'DP - Feature Selection') &
    (tracker.plans['action'] == 'drop') &
    (tracker.plans['status'] == 'planned')
)

tracker.plans.loc[mask, 'status'] = 'applied'

for var in planned_drops:
    tracker.log(
        var=var,
        stage='DP - Feature Selection',
        action='drop',
        method='feature_selection',
        before='feature present',
        after='feature removed'
    )

In [43]:
# Ver qué variables de esos findings todavía existen
vars_check = [
    'GarageYrBlt',
    'Utilities',
    'Street',
    'Condition2',
    'RoofMatl',
    'Heating',
    'GarageCond',
    'GarageQual',
    'GrLivArea',
    'GarageCars',
    'GarageArea',
    'YearBuilt',
    'TotRmsAbvGrd',
    'TotalBsmtSF',
    '1stFlrSF'
]

for var in vars_check:
    print(f'{var:15} -> {"PRESENT" if var in df_fs.columns else "DROPPED"}')

GarageYrBlt     -> DROPPED
Utilities       -> DROPPED
Street          -> DROPPED
Condition2      -> PRESENT
RoofMatl        -> PRESENT
Heating         -> PRESENT
GarageCond      -> PRESENT
GarageQual      -> PRESENT
GrLivArea       -> DROPPED
GarageCars      -> PRESENT
GarageArea      -> PRESENT
YearBuilt       -> DROPPED
TotRmsAbvGrd    -> PRESENT
TotalBsmtSF     -> DROPPED
1stFlrSF        -> DROPPED


### Related Findings

Some numeric references in the original were inconsistent with their descriptions: the temporal pair and basement pair were interchanged. Those references are not copied. Removing source columns does not correct raw inconsistencies or erase the outlier observations; their findings are not closed as data corrections here.

In [44]:
# Resolve only the near-constant findings directly addressed by this local selection.
for key, var in [("utilities_near_constant", "Utilities"), ("street_near_constant", "Street")]:
    tracker.resolve_finding(key=key, resolution=f"Se propone eliminar {var} en esta alternativa local de selección; la preparación común conserva la columna.")


In [45]:
for key, var in [
    ("condition2_near_constant", "Condition2"), ("roof_material_near_constant", "RoofMatl"),
    ("heating_near_constant", "Heating"), ("garage_condition_near_constant", "GarageCond"),
    ("garage_quality_near_constant", "GarageQual"),
]:
    tracker.resolve_finding(key=key, resolution=f"Se revisó {var} y se conservó en la alternativa local; la baja variación por sí sola no justificó eliminarla.")


In [46]:
tracker.resolve_finding(
    key="garage_cars_area_correlation",
    resolution="Se revisó la correlación entre GarageCars y GarageArea y se conservaron ambas en la alternativa local de selección.",
)


### Validation and Save Decisions

In [47]:
assert len(df_fs) == len(df_before)
pd.testing.assert_series_equal(df_fs["Id"], df_before["Id"])
pd.testing.assert_frame_equal(df_fs, df_before[df_fs.columns])
pd.testing.assert_frame_equal(df_hp, df_before)
pd.testing.assert_frame_equal(df_before, load_stage("common"))
tracker.save()
